# **Requirements**

Python 3.8.10

pythonnet 2.5.2

# **Connect to Zemax and initiate a system**

In [5]:
import base_interactive as base_interactive
import pczl

zos = base_interactive.PythonZOSConnection()
ZOSAPI = zos.ZOSAPI
TheApplication = zos.TheApplication
TheSystem = zos.TheSystem

print('Connected to OpticStudio')
# The connection should now be ready to use.  For example:
print('Serial #: ', TheApplication.SerialCode)

Connected to OpticStudio
Serial #:  20120530


# **Set System Data**

In [6]:
# Define System Explore
SysExplore = TheSystem.SystemData

## **Set Title and Notes**

In [7]:
# Set Title and Notes
SysExplore.TitleNotes.Title = "Paraxial Zoom Lens Generator"
SysExplore.TitleNotes.Notes = "Generate a paraxial zoom lens based on the calculation of the positive and negative compensated zoom lens formulas."
SysExplore.TitleNotes.Author = "Ziyi Xiong"

## **Set Aperture**

In [8]:
# Set Aperture
SysExplore.Aperture.ApertureType = ZOSAPI.SystemData.ZemaxApertureType.ImageSpaceFNum
SysExplore.Aperture.ApertureValue = 5.6

## **Set Fields**

In [9]:
# Set Fields
SysExplore.Fields.SetFieldType(ZOSAPI.SystemData.FieldType.ParaxialImageHeight)
SysExplore.Fields.ApplyFieldWizard(ZOSAPI.SystemData.FieldPattern.EqualAreaY, 9, 6.6, 0, 0, 0, True, False)

## **Set Wavelengths**

### **Via Presets**

In [10]:
# SysExplore.Wavelengths.SelectWavelengthPreset(ZOSAPI.SystemData.WavelengthPreset.FdC_Visible)

### **Customize Wavelengths**

In [11]:
# Set Wavelengths Values of Cellphone lens
num_wavelengths = SysExplore.Wavelengths.NumberOfWavelengths

if num_wavelengths == 1:
    SysExplore.Wavelengths.GetWavelength(1).Wavelength = 0.5876
    SysExplore.Wavelengths.GetWavelength(1).Weight = 24
    SysExplore.Wavelengths.AddWavelength(0.6563, 11)
    SysExplore.Wavelengths.AddWavelength(0.5461, 24)
    SysExplore.Wavelengths.AddWavelength(0.4861, 12)
    SysExplore.Wavelengths.AddWavelength(0.4360, 3)
    SysExplore.Wavelengths.AddWavelength(0.4047, 1)
    
print("Number of wavelengths after insertion: ", num_wavelengths)

Number of wavelengths after insertion:  6


### **Remove Wavelengths**

In [12]:
# if num_wavelengths > 1: [SysExplore.Wavelengths.RemoveWavelength(i) for i in range(num_wavelengths, 1, -1)]

print(list(range(3, 1, -1)))

[3, 2]


# **Set Multi-Configuration**

In [18]:
SysMCE = TheSystem.MCE

## **Add Configurations**

In [19]:
num_configs = SysMCE.NumberOfConfigurations

if num_configs == 1:
    SysMCE.AddConfiguration(False)

## **Add and Set Operands**

In [ ]:
num_operands = SysMCE.NumberOfOperands

if num_operands == 1:
    for i in range(2):
        SysMCE.AddOperand()

# Use a list to store all MC operands objects        
MC_Operand=[SysMCE.GetOperandAt(i) for i in range(1, num_operands + 1)]
MC_Operand.insert(0, None)  # to make the index start from 1

THIC = ZOSAPI.Editors.MCE.MultiConfigOperandType.THIC

# Set Surface Numbers of the Operands
for i in range(1, num_operands + 1):
    MC_Operand[i].ChangeType(THIC)
    MC_Operand[i].Param1 = i + 1  # Surface Number

# **Set Lens Data**

In [13]:
SysLDE = TheSystem.LDE

## **Add Surfaces**

In [14]:
num_surfaces = SysLDE.NumberOfSurfaces

if num_surfaces == 3:
    for i in range(4): # range(4) = [0, 1, 2, 3]
        SysLDE.AddSurface()
        
print("Number of surfaces after insertion: ", num_surfaces)

Number of surfaces after insertion:  7


## **Get and Set Surface Type**

In [15]:
# Use a list to store all surfaces objects
Surface=[SysLDE.GetSurfaceAt(i) for i in range(0, num_surfaces)] 

Paraxial_Surface = ZOSAPI.Editors.LDE.SurfaceType.Paraxial

# Change Surface Types to Paraxial
for i in range(2, num_surfaces - 1):
    st = SysLDE.GetSurfaceAt(i).GetSurfaceTypeSettings(Paraxial_Surface)
    Surface[i].ChangeType(st)

## **Set Surface Data**

### **Set Stop**

In [16]:
Surface[2].IsStop = True 

### **Set Paraxial Focal Length**

In [17]:
Paraxial_Focal_Length = ZOSAPI.Editors.LDE.SurfaceColumn.Par1

# Get the values of the PCZL parameters
pczl = pczl.PCZL(f_3 = 1.2, m_4 = 3, d_12s = 0.5, d_34s = 0.5, q = 0.2785, num_samples=101)

Surface[2].SurfaceData.Par1.DoubleValue = pczl.f_1
Surface[3].SurfaceData.Par1.DoubleValue = pczl.f_2
Surface[4].SurfaceData.Par1.DoubleValue = pczl.f_3
Surface[5].SurfaceData.Par1.DoubleValue = pczl.f_4

### **Set Comments and Thickness**

In [ ]:
Surface[1].Comment = "Dummy"
Surface[2].Comment = "Front Fixed Group"
Surface[3].Comment = "Variator"
Surface[4].Comment = "Compensator"
Surface[5].Comment = "Rear Fixed Group"

Surface[1].Thickness = 1.0